In [1]:
# CELL 1: cài đặt thư viện
# Mục đích:
# - Cài các thư viện cần cho xử lý ảnh, hash, split dữ liệu, log và xuất YAML.

!pip -q install --upgrade pip
!pip -q install pillow imagehash pandas numpy scikit-learn pyyaml tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 31.0 MB/s eta 0:00:00


In [2]:
# CELL 2: mount drive và khai báo đường dẫn
# Mục đích:
# - Mount Google Drive
# - Xác định đường dẫn dữ liệu nguồn và nơi lưu output an toàn

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# ====== CHỈNH ĐƯỜNG DẪN GỐC Ở ĐÂY NẾU CẦN ======
PROJECT_ROOT = Path("/content/drive/MyDrive/Papaya_Leaf_Disease_Classification")

# Nguồn dữ liệu
SOURCE_ORIGINAL_DIR = PROJECT_ROOT / "BDPapayaLeaf" / "Original Images"
SOURCE_LABEL_DIR    = PROJECT_ROOT / "YOLO_Bounding_Box"

# Nơi lưu output của Notebook 1
PREP_ROOT   = PROJECT_ROOT / "prepared_data_v1"
CLEAN_ROOT  = PREP_ROOT / "Dataset"
YOLO_OUT    = PREP_ROOT / "yolo_dataset"
CNN_OUT     = PREP_ROOT / "cnn_dataset"
REPORT_ROOT = PREP_ROOT / "reports"

print("PROJECT_ROOT       :", PROJECT_ROOT)
print("SOURCE_ORIGINAL_DIR:", SOURCE_ORIGINAL_DIR)
print("SOURCE_LABEL_DIR   :", SOURCE_LABEL_DIR)
print("PREP_ROOT          :", PREP_ROOT)
print("CLEAN_ROOT         :", CLEAN_ROOT)
print("YOLO_OUT           :", YOLO_OUT)
print("CNN_OUT            :", CNN_OUT)
print("REPORT_ROOT        :", REPORT_ROOT)

assert SOURCE_ORIGINAL_DIR.exists(), f"Không tìm thấy thư mục ảnh gốc: {SOURCE_ORIGINAL_DIR}"
assert SOURCE_LABEL_DIR.exists(), f"Không tìm thấy thư mục label YOLO: {SOURCE_LABEL_DIR}"

Mounted at /content/drive
PROJECT_ROOT       : /content/drive/MyDrive/Papaya_Leaf_Disease_Classification
SOURCE_ORIGINAL_DIR: /content/drive/MyDrive/Papaya_Leaf_Disease_Classification/BDPapayaLeaf/Original Images
SOURCE_LABEL_DIR   : /content/drive/MyDrive/Papaya_Leaf_Disease_Classification/YOLO_Bounding_Box
PREP_ROOT          : /content/drive/MyDrive/Papaya_Leaf_Disease_Classification/prepared_data_v1
CLEAN_ROOT         : /content/drive/MyDrive/Papaya_Leaf_Disease_Classification/prepared_data_v1/Dataset
YOLO_OUT           : /content/drive/MyDrive/Papaya_Leaf_Disease_Classification/prepared_data_v1/yolo_dataset
CNN_OUT            : /content/drive/MyDrive/Papaya_Leaf_Disease_Classification/prepared_data_v1/cnn_dataset
REPORT_ROOT        : /content/drive/MyDrive/Papaya_Leaf_Disease_Classification/prepared_data_v1/reports


In [3]:
# CELL 3: import thư viện và cấu hình chung
# Mục đích:
# - Import đầy đủ các thư viện
# - Thiết lập cấu hình chung cho toàn notebook

import os
import re
import json
import yaml
import math
import time
import shutil
import random
import hashlib
import warnings
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
from PIL import Image, ImageFile
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
import imagehash

warnings.filterwarnings("ignore")
Image.MAX_IMAGE_PIXELS = None
ImageFile.LOAD_TRUNCATED_IMAGES = False

# ====== RANDOM SEED ======
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ====== CLASS CONFIG ======
DISEASE_CLASSES = ["Anthracnose", "BacterialSpot", "Curl", "RingSpot"]
ALL_CLASSES = DISEASE_CLASSES + ["Healthy"]

CLASS_NAME_MAP = {
    "anthracnose": "Anthracnose",
    "bacterialspot": "BacterialSpot",
    "curl": "Curl",
    "healthy": "Healthy",
    "ringspot": "RingSpot",
    "ringspotdisease": "RingSpot",
    "ringspotleaf": "RingSpot",
}

YOLO_CLASS_TO_ID = {name: idx for idx, name in enumerate(DISEASE_CLASSES)}
YOLO_ID_TO_CLASS = {idx: name for name, idx in YOLO_CLASS_TO_ID.items()}

# ====== FILE CONFIG ======
VALID_IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
LABEL_EXT = ".txt"

# ====== SPLIT CONFIG ======
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

assert abs((TRAIN_RATIO + VAL_RATIO + TEST_RATIO) - 1.0) < 1e-9, "Tổng tỉ lệ split phải = 1"

# ====== CNN CROP CONFIG ======
CNN_BBOX_EXPAND_RATIO = 0.10   # mở rộng bbox mỗi phía 10%
CNN_MIN_CROP_SIZE     = 8      # bỏ crop quá bé
CNN_CROP_SAVE_EXT     = ".jpg" # crop bệnh lưu JPG
JPEG_QUALITY          = 95

# ====== DUPLICATE / CONFLICT CONFIG ======
ENABLE_PHASH_REVIEW             = True   # dùng pHash để tạo danh sách review thêm
AUTO_EXCLUDE_CROSS_CLASS_NAME   = True   # auto loại nếu cùng stem xuất hiện ở nhiều class
AUTO_EXCLUDE_CROSS_CLASS_SHA1   = True   # auto loại nếu ảnh giống hệt bytes nhưng ở nhiều class
AUTO_EXCLUDE_INTRA_CLASS_SHA1   = False  # duplicate cùng class: mặc định chỉ log, không auto loại
AUTO_EXCLUDE_AMBIGUOUS_STEMS    = True   # auto loại nếu cùng class có nhiều ảnh cùng stem hoặc nhiều label cùng stem

# ====== OUTPUT CONFIG ======
REBUILD_OUTPUT_DIRS = True   # xóa output cũ của notebook 1 để build lại sạch sẽ

print("DISEASE_CLASSES:", DISEASE_CLASSES)
print("ALL_CLASSES    :", ALL_CLASSES)
print("YOLO_CLASS_TO_ID:", YOLO_CLASS_TO_ID)

DISEASE_CLASSES: ['Anthracnose', 'BacterialSpot', 'Curl', 'RingSpot']
ALL_CLASSES    : ['Anthracnose', 'BacterialSpot', 'Curl', 'RingSpot', 'Healthy']
YOLO_CLASS_TO_ID: {'Anthracnose': 0, 'BacterialSpot': 1, 'Curl': 2, 'RingSpot': 3}


In [4]:
# CELL 4: các hàm tiện ích
# Mục đích:
# - Gom toàn bộ hàm dùng lại cho các bước scan, validate, split, crop, log

def ensure_dir(path: Path):
    path.mkdir(parents=True, exist_ok=True)

def reset_dir(path: Path):
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

def safe_relpath(path: Path, start: Path):
    try:
        return str(path.relative_to(start))
    except Exception:
        return str(path)

def normalize_class_name(name: str):
    if name is None:
        return None
    key = re.sub(r"[\s_\-]+", "", str(name).strip().lower())
    return CLASS_NAME_MAP.get(key)

def sanitize_stem(stem: str):
    stem = re.sub(r"[^\w\-\.]+", "_", str(stem).strip())
    stem = stem.strip("._")
    return stem if stem else "image"

def sha1_of_file(file_path: Path, chunk_size: int = 1024 * 1024):
    h = hashlib.sha1()
    with open(file_path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def verify_image_and_get_meta(image_path: Path):
    """
    verify ảnh + lấy width, height + phash
    """
    with Image.open(image_path) as img:
        img.verify()

    with Image.open(image_path) as img:
        img = img.convert("RGB")
        width, height = img.size
        phash = str(imagehash.phash(img))

    return width, height, phash

def list_image_files(root: Path):
    files = []
    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in VALID_IMAGE_EXTS:
            files.append(p)
    return sorted(files)

def list_label_files(root: Path):
    return sorted([p for p in root.rglob("*") if p.is_file() and p.suffix.lower() == LABEL_EXT])

def save_df(df: pd.DataFrame, path: Path, index=False):
    ensure_dir(path.parent)
    df.to_csv(path, index=index, encoding="utf-8-sig")

def save_json(obj, path: Path):
    ensure_dir(path.parent)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def format_float6(x: float):
    return f"{float(x):.6f}"

def read_text_file(path: Path):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()

def validate_and_rewrite_yolo_label(label_path: Path, expected_class_id: int):
    """
    Kiểm tra file YOLO label.
    - fatal error => file invalid, bỏ qua
    - raw class id sai => log warning nhưng vẫn remap sang expected_class_id
    Trả về:
    {
        "is_valid": bool,
        "normalized_lines": [...],
        "errors": [...],
        "warnings": [...],
        "num_boxes": int
    }
    """
    errors = []
    warnings_ = []
    normalized_lines = []

    raw_text = read_text_file(label_path)
    lines = [ln.strip() for ln in raw_text.splitlines() if ln.strip()]

    if len(lines) == 0:
        errors.append("empty_label_file")
        return {
            "is_valid": False,
            "normalized_lines": [],
            "errors": errors,
            "warnings": warnings_,
            "num_boxes": 0,
        }

    raw_ids = []

    for line_idx, line in enumerate(lines, start=1):
        parts = line.split()
        if len(parts) != 5:
            errors.append(f"line_{line_idx}: invalid_token_count={len(parts)}")
            continue

        cls_str, x_str, y_str, w_str, h_str = parts

        try:
            raw_cls = int(float(cls_str))
            x = float(x_str)
            y = float(y_str)
            w = float(w_str)
            h = float(h_str)
        except Exception:
            errors.append(f"line_{line_idx}: non_numeric_value")
            continue

        raw_ids.append(raw_cls)

        if raw_cls not in YOLO_ID_TO_CLASS:
            warnings_.append(f"line_{line_idx}: raw_class_id_out_of_range={raw_cls}")

        if not (0.0 <= x <= 1.0 and 0.0 <= y <= 1.0):
            errors.append(f"line_{line_idx}: center_out_of_range")
            continue

        if not (0.0 < w <= 1.0 and 0.0 < h <= 1.0):
            errors.append(f"line_{line_idx}: width_or_height_out_of_range")
            continue

        x1 = x - w / 2.0
        y1 = y - h / 2.0
        x2 = x + w / 2.0
        y2 = y + h / 2.0

        if x1 < 0.0 or y1 < 0.0 or x2 > 1.0 or y2 > 1.0:
            errors.append(f"line_{line_idx}: bbox_exceeds_image_bounds")
            continue

        normalized_lines.append(
            f"{expected_class_id} {format_float6(x)} {format_float6(y)} {format_float6(w)} {format_float6(h)}"
        )

    if len(set(raw_ids)) > 1:
        warnings_.append(f"mixed_raw_class_ids={sorted(set(raw_ids))}")

    is_valid = (len(errors) == 0) and (len(normalized_lines) > 0)

    return {
        "is_valid": is_valid,
        "normalized_lines": normalized_lines,
        "errors": errors,
        "warnings": warnings_,
        "num_boxes": len(normalized_lines),
    }

def validate_clean_yolo_label(label_path: Path, expected_class_id: int):
    """
    Validate lại label đã clean.
    Ở bước này class id bắt buộc phải khớp expected_class_id.
    """
    errors = []
    raw_text = read_text_file(label_path)
    lines = [ln.strip() for ln in raw_text.splitlines() if ln.strip()]

    if len(lines) == 0:
        errors.append("empty_clean_label_file")
        return False, errors

    for line_idx, line in enumerate(lines, start=1):
        parts = line.split()
        if len(parts) != 5:
            errors.append(f"line_{line_idx}: invalid_token_count={len(parts)}")
            continue

        try:
            cls_id = int(parts[0])
            x, y, w, h = map(float, parts[1:])
        except Exception:
            errors.append(f"line_{line_idx}: non_numeric_value")
            continue

        if cls_id != expected_class_id:
            errors.append(f"line_{line_idx}: class_id_mismatch={cls_id} expected={expected_class_id}")

        if not (0.0 <= x <= 1.0 and 0.0 <= y <= 1.0):
            errors.append(f"line_{line_idx}: center_out_of_range")

        if not (0.0 < w <= 1.0 and 0.0 < h <= 1.0):
            errors.append(f"line_{line_idx}: width_or_height_out_of_range")

        x1 = x - w / 2.0
        y1 = y - h / 2.0
        x2 = x + w / 2.0
        y2 = y + h / 2.0

        if x1 < 0.0 or y1 < 0.0 or x2 > 1.0 or y2 > 1.0:
            errors.append(f"line_{line_idx}: bbox_exceeds_image_bounds")

    return len(errors) == 0, errors

def yolo_to_pixel_box(x, y, w, h, img_w, img_h, expand_ratio=0.10):
    """
    YOLO normalized bbox -> pixel bbox, có nới thêm context
    """
    bw = w * img_w
    bh = h * img_h
    cx = x * img_w
    cy = y * img_h

    x1 = cx - bw / 2.0
    y1 = cy - bh / 2.0
    x2 = cx + bw / 2.0
    y2 = cy + bh / 2.0

    expand_w = bw * expand_ratio
    expand_h = bh * expand_ratio

    x1 -= expand_w
    y1 -= expand_h
    x2 += expand_w
    y2 += expand_h

    x1 = max(0, int(math.floor(x1)))
    y1 = max(0, int(math.floor(y1)))
    x2 = min(img_w, int(math.ceil(x2)))
    y2 = min(img_h, int(math.ceil(y2)))

    return x1, y1, x2, y2

def read_clean_label_rows(label_path: Path):
    rows = []
    lines = [ln.strip() for ln in read_text_file(label_path).splitlines() if ln.strip()]
    for ln in lines:
        cls_id, x, y, w, h = ln.split()
        rows.append((int(cls_id), float(x), float(y), float(w), float(h)))
    return rows

def unique_output_filename(class_name: str, stem: str, suffix: str, counter: int):
    stem = sanitize_stem(stem)
    return f"{class_name}_{counter:05d}_{stem}{suffix.lower()}"

def robust_grouped_split(df: pd.DataFrame, label_col: str, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15, seed=42):
    """
    fallback chia theo từng lớp nếu stratify thất bại
    """
    rng = random.Random(seed)
    train_parts, val_parts, test_parts = [], [], []

    for label, group in df.groupby(label_col):
        group = group.sample(frac=1.0, random_state=seed).reset_index(drop=True)
        n = len(group)

        if n == 1:
            n_train, n_val, n_test = 1, 0, 0
        elif n == 2:
            n_train, n_val, n_test = 1, 1, 0
        elif n == 3:
            n_train, n_val, n_test = 2, 1, 0
        else:
            n_test = max(1, int(round(n * test_ratio)))
            n_val  = max(1, int(round(n * val_ratio)))
            n_train = n - n_val - n_test

            while n_train < 1 and n_val > 1:
                n_val -= 1
                n_train += 1

            while n_train < 1 and n_test > 1:
                n_test -= 1
                n_train += 1

            if n_train < 1:
                n_train = 1
                if n_val >= n_test and n_val > 0:
                    n_val -= 1
                elif n_test > 0:
                    n_test -= 1

        train_parts.append(group.iloc[:n_train])
        val_parts.append(group.iloc[n_train:n_train+n_val])
        test_parts.append(group.iloc[n_train+n_val:n_train+n_val+n_test])

    train_df = pd.concat(train_parts, ignore_index=True) if train_parts else pd.DataFrame(columns=df.columns)
    val_df   = pd.concat(val_parts,   ignore_index=True) if val_parts   else pd.DataFrame(columns=df.columns)
    test_df  = pd.concat(test_parts,  ignore_index=True) if test_parts  else pd.DataFrame(columns=df.columns)

    train_df = train_df.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    val_df   = val_df.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    test_df  = test_df.sample(frac=1.0, random_state=seed).reset_index(drop=True)

    return train_df, val_df, test_df

def safe_two_stage_split(df: pd.DataFrame, label_col: str, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15, seed=42):
    """
    Ưu tiên stratify bằng train_test_split.
    Nếu lỗi do ít mẫu/lớp quá nhỏ thì fallback sang grouped split.
    """
    assert len(df) > 0, "DataFrame split rỗng"

    try:
        train_df, temp_df = train_test_split(
            df,
            test_size=(1.0 - train_ratio),
            random_state=seed,
            shuffle=True,
            stratify=df[label_col]
        )

        temp_ratio = val_ratio + test_ratio
        rel_test_ratio = test_ratio / temp_ratio if temp_ratio > 0 else 0.5

        # temp có thể quá nhỏ cho stratify lần 2
        temp_counts = temp_df[label_col].value_counts()
        if len(temp_counts) > 0 and temp_counts.min() >= 2 and temp_df[label_col].nunique() >= 2:
            val_df, test_df = train_test_split(
                temp_df,
                test_size=rel_test_ratio,
                random_state=seed,
                shuffle=True,
                stratify=temp_df[label_col]
            )
            method = "stratified_train_test_split"
        else:
            val_df, test_df = robust_grouped_split(
                temp_df,
                label_col=label_col,
                train_ratio=(val_ratio / temp_ratio) if temp_ratio > 0 else 0.5,
                val_ratio=(test_ratio / temp_ratio) if temp_ratio > 0 else 0.5,
                test_ratio=0.0,
                seed=seed
            )
            method = "hybrid_split(stratified_first_stage + grouped_second_stage)"

    except Exception as e:
        train_df, val_df, test_df = robust_grouped_split(
            df,
            label_col=label_col,
            train_ratio=train_ratio,
            val_ratio=val_ratio,
            test_ratio=test_ratio,
            seed=seed
        )
        method = f"grouped_fallback_due_to_{type(e).__name__}"

    return (
        train_df.reset_index(drop=True),
        val_df.reset_index(drop=True),
        test_df.reset_index(drop=True),
        method
    )

def simple_random_split_single_class(df: pd.DataFrame, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15, seed=42):
    df = df.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    n = len(df)
    n_train = int(round(n * train_ratio))
    n_val   = int(round(n * val_ratio))

    if n >= 3:
        n_train = max(1, n_train)
        n_val   = max(1, n_val)
        while n_train + n_val >= n:
            if n_train > n_val and n_train > 1:
                n_train -= 1
            elif n_val > 1:
                n_val -= 1
            else:
                break

    n_test = n - n_train - n_val
    train_df = df.iloc[:n_train].reset_index(drop=True)
    val_df   = df.iloc[n_train:n_train+n_val].reset_index(drop=True)
    test_df  = df.iloc[n_train+n_val:].reset_index(drop=True)
    return train_df, val_df, test_df

In [6]:
# CELL 5: quét dữ liệu nguồn
# Mục đích:
# - Đọc toàn bộ ảnh nguồn từ Original Images
# - Đọc toàn bộ label nguồn từ YOLO_Bounding_Box
# - Tạo manifest ban đầu để các cell sau sử dụng

if REBUILD_OUTPUT_DIRS:
    reset_dir(PREP_ROOT)
    ensure_dir(REPORT_ROOT)
else:
    ensure_dir(PREP_ROOT)
    ensure_dir(REPORT_ROOT)

image_records = []
image_scan_issues = []

print("=== SCAN SOURCE IMAGES ===")
original_subdirs = [p for p in SOURCE_ORIGINAL_DIR.iterdir() if p.is_dir()]
print("Found image class folders:", [p.name for p in original_subdirs])

for class_dir in sorted(original_subdirs):
    canonical_class = normalize_class_name(class_dir.name)

    if canonical_class is None:
        image_scan_issues.append({
            "issue_type": "unknown_image_class_folder",
            "folder_name": class_dir.name,
            "folder_path": str(class_dir)
        })
        continue

    class_images = list_image_files(class_dir)
    print(f"[{canonical_class}] found {len(class_images)} image files")

    for img_path in tqdm(class_images, desc=f"Scan images - {canonical_class}", leave=False):
        record = {
            "source_type": "image",
            "source_class_folder": class_dir.name,
            "canonical_class": canonical_class,
            "source_path": str(img_path),
            "source_relpath": safe_relpath(img_path, PROJECT_ROOT),
            "filename": img_path.name,
            "stem": img_path.stem,
            "stem_key": img_path.stem.strip().lower(),
            "suffix": img_path.suffix.lower(),
            "file_size_bytes": img_path.stat().st_size if img_path.exists() else None,
            "is_broken": False,
            "width": None,
            "height": None,
            "sha1": None,
            "phash": None,
            "error_message": None,
        }

        try:
            width, height, phash = verify_image_and_get_meta(img_path)
            sha1 = sha1_of_file(img_path)

            record["width"] = width
            record["height"] = height
            record["phash"] = phash
            record["sha1"] = sha1

        except Exception as e:
            record["is_broken"] = True
            record["error_message"] = str(e)
            image_scan_issues.append({
                "issue_type": "broken_image",
                "canonical_class": canonical_class,
                "source_path": str(img_path),
                "error_message": str(e)
            })

        image_records.append(record)

images_df = pd.DataFrame(image_records)
image_scan_issues_df = pd.DataFrame(image_scan_issues)

print("\n=== SCAN SOURCE LABELS ===")
label_records = []
label_scan_issues = []

label_subdirs = [p for p in SOURCE_LABEL_DIR.iterdir() if p.is_dir()]
print("Found label class folders:", [p.name for p in label_subdirs])

for class_dir in sorted(label_subdirs):
    canonical_class = normalize_class_name(class_dir.name)

    if canonical_class is None:
        label_scan_issues.append({
            "issue_type": "unknown_label_class_folder",
            "folder_name": class_dir.name,
            "folder_path": str(class_dir)
        })
        continue

    if canonical_class == "Healthy":
        label_scan_issues.append({
            "issue_type": "unexpected_healthy_label_folder",
            "folder_name": class_dir.name,
            "folder_path": str(class_dir)
        })
        continue

    label_files = list_label_files(class_dir)
    print(f"[{canonical_class}] found {len(label_files)} label files")

    for lbl_path in tqdm(label_files, desc=f"Scan labels - {canonical_class}", leave=False):
        label_records.append({
            "source_type": "label",
            "source_class_folder": class_dir.name,
            "canonical_class": canonical_class,
            "label_path": str(lbl_path),
            "label_relpath": safe_relpath(lbl_path, PROJECT_ROOT),
            "filename": lbl_path.name,
            "stem": lbl_path.stem,
            "stem_key": lbl_path.stem.strip().lower(),
            "suffix": lbl_path.suffix.lower(),
            "file_size_bytes": lbl_path.stat().st_size if lbl_path.exists() else None,
        })

labels_df = pd.DataFrame(label_records)
label_scan_issues_df = pd.DataFrame(label_scan_issues)

# Lưu manifest
save_df(images_df, REPORT_ROOT / "01_source_images_manifest.csv")
save_df(labels_df, REPORT_ROOT / "02_source_labels_manifest.csv")
save_df(image_scan_issues_df, REPORT_ROOT / "03_image_scan_issues.csv")
save_df(label_scan_issues_df, REPORT_ROOT / "04_label_scan_issues.csv")

print("\n=== SOURCE SUMMARY ===")
print("Total source images:", len(images_df))
print("Broken images      :", int(images_df["is_broken"].sum()) if len(images_df) else 0)
print("Total source labels:", len(labels_df))
display(images_df.groupby("canonical_class").size().reset_index(name="num_images"))
display(labels_df.groupby("canonical_class").size().reset_index(name="num_labels"))

=== SCAN SOURCE IMAGES ===
Found image class folders: ['Anthracnose', 'Curl', 'Healthy', 'BacterialSpot', 'RingSpot']
[Anthracnose] found 355 image files


Scan images - Anthracnose:   0%|          | 0/355 [00:00<?, ?it/s]

[BacterialSpot] found 458 image files


Scan images - BacterialSpot:   0%|          | 0/458 [00:00<?, ?it/s]

[Curl] found 585 image files


...

[RingSpot] found 533 label files


Scan labels - RingSpot:   0%|          | 0/533 [00:00<?, ?it/s]


=== SOURCE SUMMARY ===
Total source images: 2159
Broken images      : 0
Total source labels: 1710


,canonical_class,num_images
0,Anthracnose,355
1,BacterialSpot,458
2,Curl,585
3,Healthy,228
4,RingSpot,533


,canonical_class,num_labels
0,Anthracnose,355
1,BacterialSpot,458
2,Curl,364
3,RingSpot,533


In [7]:
# CELL 6: phát hiện conflict và duplicate
# Mục đích:
# - Tìm ảnh trùng tên giữa nhiều class
# - Tìm ảnh giống hệt bytes (SHA1) giữa nhiều class
# - Tìm duplicate trong cùng class
# - Tìm ambiguous stem cho image/label
# - Tạo auto-exclude manifest an toàn

valid_images_df = images_df[images_df["is_broken"] == False].copy()

# ===== 1) conflict theo stem giữa nhiều class =====
name_conflict_keys = (
    valid_images_df.groupby("stem_key")["canonical_class"]
    .nunique()
    .reset_index(name="n_classes")
)
name_conflict_keys = name_conflict_keys[name_conflict_keys["n_classes"] > 1]["stem_key"].tolist()

name_conflicts_df = valid_images_df[valid_images_df["stem_key"].isin(name_conflict_keys)].copy()
save_df(name_conflicts_df, REPORT_ROOT / "05_name_conflicts_cross_class.csv")

# ===== 2) conflict theo SHA1 giữa nhiều class =====
sha_conflict_keys = (
    valid_images_df.dropna(subset=["sha1"])
    .groupby("sha1")["canonical_class"]
    .nunique()
    .reset_index(name="n_classes")
)
sha_conflict_keys = sha_conflict_keys[sha_conflict_keys["n_classes"] > 1]["sha1"].tolist()

sha_conflicts_df = valid_images_df[valid_images_df["sha1"].isin(sha_conflict_keys)].copy()
save_df(sha_conflicts_df, REPORT_ROOT / "06_sha1_conflicts_cross_class.csv")

# ===== 3) review bằng pHash giữa nhiều class (chỉ review, không auto-remove mặc định) =====
if ENABLE_PHASH_REVIEW:
    phash_conflict_keys = (
        valid_images_df.dropna(subset=["phash"])
        .groupby("phash")["canonical_class"]
        .nunique()
        .reset_index(name="n_classes")
    )
    phash_conflict_keys = phash_conflict_keys[phash_conflict_keys["n_classes"] > 1]["phash"].tolist()
    phash_review_df = valid_images_df[valid_images_df["phash"].isin(phash_conflict_keys)].copy()
else:
    phash_review_df = pd.DataFrame(columns=valid_images_df.columns)

save_df(phash_review_df, REPORT_ROOT / "07_phash_review_cross_class.csv")

# ===== 4) duplicate cùng class theo SHA1 =====
same_class_sha_dups = []
for (cls_name, sha1), group in valid_images_df.dropna(subset=["sha1"]).groupby(["canonical_class", "sha1"]):
    if len(group) > 1:
        group = group.sort_values("source_path").reset_index(drop=True)
        keep_path = group.loc[0, "source_path"]
        for idx in range(1, len(group)):
            same_class_sha_dups.append({
                "canonical_class": cls_name,
                "sha1": sha1,
                "kept_source_path": keep_path,
                "duplicate_source_path": group.loc[idx, "source_path"],
                "duplicate_filename": group.loc[idx, "filename"],
            })

same_class_sha_dups_df = pd.DataFrame(same_class_sha_dups)
save_df(same_class_sha_dups_df, REPORT_ROOT / "08_intra_class_sha1_duplicates.csv")

# ===== 5) ambiguous image stem trong cùng class bệnh =====
ambiguous_image_stems_df = (
    valid_images_df[valid_images_df["canonical_class"].isin(DISEASE_CLASSES)]
    .groupby(["canonical_class", "stem_key"])
    .size()
    .reset_index(name="num_images_with_same_stem")
)
ambiguous_image_stems_df = ambiguous_image_stems_df[ambiguous_image_stems_df["num_images_with_same_stem"] > 1]
save_df(ambiguous_image_stems_df, REPORT_ROOT / "09_ambiguous_image_stems_within_class.csv")

# ===== 6) ambiguous label stem trong cùng class bệnh =====
ambiguous_label_stems_df = (
    labels_df.groupby(["canonical_class", "stem_key"])
    .size()
    .reset_index(name="num_labels_with_same_stem")
)
ambiguous_label_stems_df = ambiguous_label_stems_df[ambiguous_label_stems_df["num_labels_with_same_stem"] > 1]
save_df(ambiguous_label_stems_df, REPORT_ROOT / "10_ambiguous_label_stems_within_class.csv")

ambiguous_image_keys = set(
    zip(ambiguous_image_stems_df["canonical_class"], ambiguous_image_stems_df["stem_key"])
)
ambiguous_label_keys = set(
    zip(ambiguous_label_stems_df["canonical_class"], ambiguous_label_stems_df["stem_key"])
)

# ===== 7) gom lý do auto-exclude =====
exclude_reason_map = defaultdict(set)

if AUTO_EXCLUDE_CROSS_CLASS_NAME and len(name_conflicts_df) > 0:
    for p in name_conflicts_df["source_path"].tolist():
        exclude_reason_map[p].add("cross_class_name_conflict")

if AUTO_EXCLUDE_CROSS_CLASS_SHA1 and len(sha_conflicts_df) > 0:
    for sha1, group in sha_conflicts_df.groupby("sha1"):
        classes_in_group = group["canonical_class"].values
        if "BacterialSpot" in classes_in_group:
            # Nếu có trùng lặp và một trong số đó là BacterialSpot -> Giữ BacterialSpot, loại bỏ các lớp khác
            for row in group.itertuples(index=False):
                if row.canonical_class != "BacterialSpot":
                    exclude_reason_map[row.source_path].add("cross_class_sha1_conflict_kept_BacterialSpot")
        else:
            # Nếu trùng lặp giữa các lớp khác (không có BacterialSpot) -> Loại bỏ hết như cũ
            for p in group["source_path"]:
                exclude_reason_map[p].add("cross_class_sha1_conflict")

if AUTO_EXCLUDE_INTRA_CLASS_SHA1 and len(same_class_sha_dups_df) > 0:
    for p in same_class_sha_dups_df["duplicate_source_path"].tolist():
        exclude_reason_map[p].add("intra_class_sha1_duplicate")

if AUTO_EXCLUDE_AMBIGUOUS_STEMS:
    for row in valid_images_df.itertuples(index=False):
        key = (row.canonical_class, row.stem_key)
        if key in ambiguous_image_keys:
            exclude_reason_map[row.source_path].add("ambiguous_image_stem_within_class")
        if row.canonical_class in DISEASE_CLASSES and key in ambiguous_label_keys:
            exclude_reason_map[row.source_path].add("ambiguous_label_stem_within_class")

auto_exclude_records = []
for row in valid_images_df.itertuples(index=False):
    reasons = sorted(list(exclude_reason_map.get(row.source_path, [])))
    if reasons:
        auto_exclude_records.append({
            "canonical_class": row.canonical_class,
            "source_path": row.source_path,
            "filename": row.filename,
            "stem_key": row.stem_key,
            "sha1": row.sha1,
            "exclude_reasons": " | ".join(reasons),
        })

auto_exclude_df = pd.DataFrame(auto_exclude_records)
save_df(auto_exclude_df, REPORT_ROOT / "11_auto_exclude_manifest.csv")

print("=== CONFLICT SUMMARY ===")
print("Cross-class name conflicts :", len(name_conflicts_df))
print("Cross-class SHA1 conflicts :", len(sha_conflicts_df))
print("Cross-class pHash review   :", len(phash_review_df))
print("Intra-class SHA1 duplicates:", len(same_class_sha_dups_df))
print("Ambiguous image stems      :", len(ambiguous_image_stems_df))
print("Ambiguous label stems      :", len(ambiguous_label_stems_df))
print("Auto-excluded source images:", len(auto_exclude_df))

if len(auto_exclude_df) > 0:
    display(auto_exclude_df.head(20))

=== CONFLICT SUMMARY ===
Cross-class name conflicts : 0
Cross-class SHA1 conflicts : 448
Cross-class pHash review   : 448
Intra-class SHA1 duplicates: 78
Ambiguous image stems      : 0
Ambiguous label stems      : 0
Auto-excluded source images: 224


,canonical_class,source_path,filename,stem_key,sha1,exclude_reasons
0,Curl,/content/drive/MyDrive/Papaya_Leaf_Disease_Cla...,Curl(319).jpg,curl(319),9430c032d118f98d69381d00023042e2ad19f7cb,cross_class_sha1_conflict_kept_BacterialSpot
1,Curl,/content/drive/MyDrive/Papaya_Leaf_Disease_Cla...,Curl(325).jpg,curl(325),db7764e9e599d251078175c953c46f56672667b4,cross_class_sha1_conflict_kept_BacterialSpot
2,Curl,/content/drive/MyDrive/Papaya_Leaf_Disease_Cla...,Curl(326).jpg,curl(326),83c3cdff31225fa96dd3230c8298aa3d8c0a4961,cross_class_sha1_conflict_kept_BacterialSpot
3,Curl,/content/drive/MyDrive/Papaya_Leaf_Disease_Cla...,Curl(364).jpg,curl(364),1fba64203e594ea98c866a52f5e53b1c73784a10,cross_class_sha1_conflict_kept_BacterialSpot
4,Curl,/content/drive/MyDrive/Papaya_Leaf_Disease_Cla...,Curl(365).jpg,curl(365),99c6c5a6e1d098b1da0372fdf2d8968200f34983,cross_class_sha1_conflict_kept_BacterialSpot
5,Curl,/content/drive/MyDrive/Papaya_Leaf_Disease_Cla...,Curl(366).jpg,curl(366),988c3f957e06014d884b564e45fb48c74a72f35d,cross_class_sha1_conflict_kept_BacterialSpot
6,Curl,/content/drive/MyDrive/Papaya_Leaf_Disease_Cla...,Curl(367).jpg,curl(367),8657b635c73e12a47721c42e4facfb31b70a16b0,cross_class_sha1_conflict_kept_BacterialSpot
7,Curl,/content/drive/MyDrive/Papaya_Leaf_Disease_Cla...,Curl(368).jpg,curl(368),bd4a7962a31c83e9c82aafe7d56859fb76a56d6b,cross_class_sha1_conflict_kept_BacterialSpot
8,Curl,/content/drive/MyDrive/Papaya_Leaf_Disease_Cla...,Curl(369).jpg,curl(369),ca44665aa0372b59cb26f59897301acf102053e8,cross_class_sha1_conflict_kept_BacterialSpot
9,Curl,/content/drive/MyDrive/Papaya_Leaf_Disease_Cla...,Curl(370).jpg,curl(370),332101bb839229c9d17919b564de794fe9523555,cross_class_sha1_conflict_kept_BacterialSpot


In [8]:
# CELL 7: tạo cleaned canonical dataset
# Mục đích:
# - Tạo dataset sạch, không đụng dữ liệu gốc
# - 4 lớp bệnh có Images + Labels
# - Healthy chỉ có Images
# - Remap label sang class id chuẩn
# - TỰ ĐỘNG CẮT XÉN (CLIP/CLAMP) TỌA ĐỘ BỊ VƯỢT GIỚI HẠN
# - Log các ảnh/label bị bỏ qua

# =====================================================================
# HÀM MỚI: Tự động kiểm tra và cắt xén (clip) tọa độ thay vì báo lỗi
# =====================================================================
def process_and_clip_yolo_label(label_path, expected_class_id):
    is_valid = True
    errors = []
    warnings = []
    normalized_lines = []
    num_boxes = 0

    try:
        with open(label_path, "r", encoding="utf-8") as f:
            lines = f.readlines()

        for idx, line in enumerate(lines):
            line = line.strip()
            if not line:
                continue

            parts = line.split()
            if len(parts) < 5:
                warnings.append(f"Dòng {idx+1} thiếu dữ liệu: bị bỏ qua.")
                continue

            try:
                # Đọc dữ liệu thô
                c_id = int(parts[0])
                x_c = float(parts[1])
                y_c = float(parts[2])
                w = float(parts[3])
                h = float(parts[4])
            except ValueError:
                warnings.append(f"Dòng {idx+1} chứa ký tự lạ: bị bỏ qua.")
                continue

            # 1. Chuyển từ (tâm x, tâm y, w, h) sang dạng 2 góc (Min-Max)
            x_min = x_c - (w / 2.0)
            x_max = x_c + (w / 2.0)
            y_min = y_c - (h / 2.0)
            y_max = y_c + (h / 2.0)

            # 2. CẮT XÉN (Clamp) ép vào giới hạn [0.0, 1.0]
            new_xmin = max(0.0, min(1.0, x_min))
            new_xmax = max(0.0, min(1.0, x_max))
            new_ymin = max(0.0, min(1.0, y_min))
            new_ymax = max(0.0, min(1.0, y_max))

            # 3. Chuyển ngược lại về format YOLO chuẩn
            new_w = new_xmax - new_xmin
            new_h = new_ymax - new_ymin
            new_xc = new_xmin + (new_w / 2.0)
            new_yc = new_ymin + (new_h / 2.0)

            # Nếu box bị cắt thành diện tích = 0 (nằm hoàn toàn ngoài rìa)
            if new_w <= 0 or new_h <= 0:
                warnings.append(f"Box dòng {idx+1} bị loại vì nằm ngoài ảnh.")
                continue

            # Nếu kích thước bị thay đổi, ghi nhận Warning nhưng KHÔNG báo lỗi
            if new_w != w or new_h != h or new_xc != x_c or new_yc != y_c:
                warnings.append(f"Box dòng {idx+1} đã được tự động cắt xén (clip) vừa ảnh.")

            # Ghi lại kết quả với 6 chữ số thập phân, gán luôn class_id chuẩn
            normalized_line = f"{expected_class_id} {new_xc:.6f} {new_yc:.6f} {new_w:.6f} {new_h:.6f}"
            normalized_lines.append(normalized_line)
            num_boxes += 1

        # Trừ khi file không có box nào hợp lệ, nếu không file này vẫn được xem là VALID
        if num_boxes == 0:
            is_valid = False
            errors.append("File nhãn rỗng hoặc các box đều nằm ngoài ảnh.")

    except Exception as e:
        is_valid = False
        errors.append(f"Lỗi đọc file: {str(e)}")

    return {
        "is_valid": is_valid,
        "errors": errors,
        "warnings": warnings,
        "normalized_lines": normalized_lines,
        "num_boxes": num_boxes
    }

# =====================================================================
# LOGIC BUILD DATASET
# =====================================================================

if REBUILD_OUTPUT_DIRS:
    reset_dir(CLEAN_ROOT)
else:
    ensure_dir(CLEAN_ROOT)

for cls_name in ALL_CLASSES:
    ensure_dir(CLEAN_ROOT / cls_name / "Images")
    if cls_name in DISEASE_CLASSES:
        ensure_dir(CLEAN_ROOT / cls_name / "Labels")

# lookup label duy nhất theo (class, stem_key)
label_lookup = {}
for row in labels_df.sort_values(["canonical_class", "label_path"]).itertuples(index=False):
    key = (row.canonical_class, row.stem_key)
    if key not in label_lookup:
        label_lookup[key] = row.label_path

auto_exclude_paths = set(auto_exclude_df["source_path"].tolist())

clean_manifest = []
clean_skip_records = []
clean_label_warning_records = []
class_copy_counter = Counter()

for row in tqdm(
    images_df.sort_values(["canonical_class", "source_path"]).itertuples(index=False),
    total=len(images_df),
    desc="Build cleaned dataset"
):
    src_path = Path(row.source_path)
    cls_name = row.canonical_class

    # bỏ class lạ
    if cls_name not in ALL_CLASSES:
        clean_skip_records.append({
            "issue_type": "unknown_canonical_class",
            "canonical_class": cls_name,
            "source_path": str(src_path),
        })
        continue

    # bỏ ảnh hỏng
    if row.is_broken:
        clean_skip_records.append({
            "issue_type": "broken_image",
            "canonical_class": cls_name,
            "source_path": str(src_path),
            "detail": row.error_message,
        })
        continue

    # bỏ ảnh conflict/duplicate/ambiguous theo rule
    if str(src_path) in auto_exclude_paths:
        reasons = auto_exclude_df.loc[auto_exclude_df["source_path"] == str(src_path), "exclude_reasons"].tolist()
        clean_skip_records.append({
            "issue_type": "auto_excluded_source_image",
            "canonical_class": cls_name,
            "source_path": str(src_path),
            "detail": reasons[0] if reasons else None,
        })
        continue

    # ảnh Healthy: chỉ copy image
    if cls_name == "Healthy":
        class_copy_counter[cls_name] += 1
        out_img_name = unique_output_filename(cls_name, src_path.stem, src_path.suffix, class_copy_counter[cls_name])
        out_img_path = CLEAN_ROOT / cls_name / "Images" / out_img_name

        shutil.copy2(src_path, out_img_path)

        clean_manifest.append({
            "canonical_class": cls_name,
            "split_role": "source_clean",
            "clean_image_path": str(out_img_path),
            "clean_label_path": None,
            "source_image_path": str(src_path),
            "source_label_path": None,
            "filename": out_img_name,
            "num_boxes": 0,
            "width": row.width,
            "height": row.height,
        })
        continue

    # ảnh bệnh: cần label hợp lệ
    key = (cls_name, row.stem_key)
    label_path_str = label_lookup.get(key, None)

    if label_path_str is None:
        clean_skip_records.append({
            "issue_type": "missing_label_for_disease_image",
            "canonical_class": cls_name,
            "source_path": str(src_path),
            "detail": f"key={key}",
        })
        continue

    label_path = Path(label_path_str)
    expected_class_id = YOLO_CLASS_TO_ID[cls_name]

    # ==> GỌI HÀM CẮT XÉN (CLIP) MỚI TẠI ĐÂY <==
    label_check = process_and_clip_yolo_label(label_path, expected_class_id=expected_class_id)

    if not label_check["is_valid"]:
        clean_skip_records.append({
            "issue_type": "invalid_source_label",
            "canonical_class": cls_name,
            "source_path": str(src_path),
            "label_path": str(label_path),
            "detail": " | ".join(label_check["errors"]) if label_check["errors"] else None,
        })
        continue

    if label_check["warnings"]:
        clean_label_warning_records.append({
            "canonical_class": cls_name,
            "source_path": str(src_path),
            "label_path": str(label_path),
            "warnings": " | ".join(label_check["warnings"])
        })

    class_copy_counter[cls_name] += 1
    out_img_name = unique_output_filename(cls_name, src_path.stem, src_path.suffix, class_copy_counter[cls_name])
    out_lbl_name = Path(out_img_name).stem + ".txt"

    out_img_path = CLEAN_ROOT / cls_name / "Images" / out_img_name
    out_lbl_path = CLEAN_ROOT / cls_name / "Labels" / out_lbl_name

    shutil.copy2(src_path, out_img_path)
    with open(out_lbl_path, "w", encoding="utf-8") as f:
        f.write("\n".join(label_check["normalized_lines"]) + "\n")

    clean_manifest.append({
        "canonical_class": cls_name,
        "split_role": "source_clean",
        "clean_image_path": str(out_img_path),
        "clean_label_path": str(out_lbl_path),
        "source_image_path": str(src_path),
        "source_label_path": str(label_path),
        "filename": out_img_name,
        "num_boxes": label_check["num_boxes"],
        "width": row.width,
        "height": row.height,
    })

clean_manifest_df = pd.DataFrame(clean_manifest)
clean_skip_df = pd.DataFrame(clean_skip_records)
clean_label_warning_df = pd.DataFrame(clean_label_warning_records)

save_df(clean_manifest_df, REPORT_ROOT / "12_clean_manifest.csv")
save_df(clean_skip_df, REPORT_ROOT / "13_clean_skip_records.csv")
save_df(clean_label_warning_df, REPORT_ROOT / "14_clean_label_warnings.csv")

print("=== CLEAN DATASET SUMMARY ===")
print("Clean samples kept :", len(clean_manifest_df))
print("Clean skips        :", len(clean_skip_df))
print("Label warnings     :", len(clean_label_warning_df))

if len(clean_manifest_df) > 0:
    display(clean_manifest_df.groupby("canonical_class").size().reset_index(name="num_clean_samples"))
if len(clean_skip_df) > 0:
    display(clean_skip_df.groupby(["canonical_class", "issue_type"]).size().reset_index(name="count").head(30))

Build cleaned dataset:   0%|          | 0/2159 [00:00<?, ?it/s]

=== CLEAN DATASET SUMMARY ===
Clean samples kept : 1935
Clean skips        : 224
Label warnings     : 1234


,canonical_class,num_clean_samples
0,Anthracnose,355
1,BacterialSpot,458
2,Curl,361
3,Healthy,228
4,RingSpot,533


,canonical_class,issue_type,count
0,Curl,auto_excluded_source_image,224


In [9]:
# CELL 8: kiểm tra lại cleaned labels
# Mục đích:
# - Quét lại toàn bộ label đã clean để xác nhận output thực sự sạch

clean_label_validation_records = []

for cls_name in DISEASE_CLASSES:
    expected_class_id = YOLO_CLASS_TO_ID[cls_name]
    label_dir = CLEAN_ROOT / cls_name / "Labels"

    for lbl_path in sorted(label_dir.glob("*.txt")):
        is_valid, errors = validate_clean_yolo_label(lbl_path, expected_class_id=expected_class_id)
        clean_label_validation_records.append({
            "canonical_class": cls_name,
            "clean_label_path": str(lbl_path),
            "is_valid": is_valid,
            "errors": " | ".join(errors) if errors else None,
        })

clean_label_validation_df = pd.DataFrame(clean_label_validation_records)
save_df(clean_label_validation_df, REPORT_ROOT / "15_clean_label_validation.csv")

num_invalid_clean_labels = int((clean_label_validation_df["is_valid"] == False).sum()) if len(clean_label_validation_df) else 0

print("=== CLEAN LABEL VALIDATION ===")
print("Total cleaned labels :", len(clean_label_validation_df))
print("Invalid cleaned labels:", num_invalid_clean_labels)

if num_invalid_clean_labels > 0:
    display(clean_label_validation_df[clean_label_validation_df["is_valid"] == False].head(20))
else:
    print("Tất cả cleaned labels đều hợp lệ.")

=== CLEAN LABEL VALIDATION ===
Total cleaned labels : 1707
Invalid cleaned labels: 158


,canonical_class,clean_label_path,is_valid,errors
12,Anthracnose,/content/drive/MyDrive/Papaya_Leaf_Disease_Cla...,False,line_22: bbox_exceeds_image_bounds
157,Anthracnose,/content/drive/MyDrive/Papaya_Leaf_Disease_Cla...,False,line_4: bbox_exceeds_image_bounds
197,Anthracnose,/content/drive/MyDrive/Papaya_Leaf_Disease_Cla...,False,line_10: bbox_exceeds_image_bounds
200,Anthracnose,/content/drive/MyDrive/Papaya_Leaf_Disease_Cla...,False,line_51: bbox_exceeds_image_bounds
203,Anthracnose,/content/drive/MyDrive/Papaya_Leaf_Disease_Cla...,False,line_25: bbox_exceeds_image_bounds
340,Anthracnose,/content/drive/MyDrive/Papaya_Leaf_Disease_Cla...,False,line_12: bbox_exceeds_image_bounds
371,BacterialSpot,/content/drive/MyDrive/Papaya_Leaf_Disease_Cla...,False,line_1: bbox_exceeds_image_bounds
390,BacterialSpot,/content/drive/MyDrive/Papaya_Leaf_Disease_Cla...,False,line_1: bbox_exceeds_image_bounds
396,BacterialSpot,/content/drive/MyDrive/Papaya_Leaf_Disease_Cla...,False,line_1: bbox_exceeds_image_bounds
397,BacterialSpot,/content/drive/MyDrive/Papaya_Leaf_Disease_Cla...,False,line_1: bbox_exceeds_image_bounds


In [10]:
# CELL 9: tạo YOLO train/val/test dataset
# Mục đích:
# - Từ cleaned disease dataset, tạo split train/val/test cho YOLO
# - Tạo cấu trúc images/labels chuẩn
# - Xuất dataset.yaml

if REBUILD_OUTPUT_DIRS:
    reset_dir(YOLO_OUT)
else:
    ensure_dir(YOLO_OUT)

for part in ["train", "val", "test"]:
    ensure_dir(YOLO_OUT / "images" / part)
    ensure_dir(YOLO_OUT / "labels" / part)

# Chỉ disease images mới vào YOLO detection dataset
clean_disease_df = clean_manifest_df[clean_manifest_df["canonical_class"].isin(DISEASE_CLASSES)].copy().reset_index(drop=True)
assert len(clean_disease_df) > 0, "Không có dữ liệu disease sạch để tạo YOLO dataset."

yolo_train_df, yolo_val_df, yolo_test_df, yolo_split_method = safe_two_stage_split(
    clean_disease_df,
    label_col="canonical_class",
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
    test_ratio=TEST_RATIO,
    seed=SEED
)

yolo_train_df["split"] = "train"
yolo_val_df["split"]   = "val"
yolo_test_df["split"]  = "test"

yolo_split_df = pd.concat([yolo_train_df, yolo_val_df, yolo_test_df], ignore_index=True)
save_df(yolo_split_df, REPORT_ROOT / "16_yolo_split_manifest_pre_copy.csv")

copy_records = []

for row in tqdm(yolo_split_df.itertuples(index=False), total=len(yolo_split_df), desc="Copy YOLO split files"):
    split_name = row.split
    src_img = Path(row.clean_image_path)
    src_lbl = Path(row.clean_label_path)

    dst_img = YOLO_OUT / "images" / split_name / src_img.name
    dst_lbl = YOLO_OUT / "labels" / split_name / src_lbl.name

    shutil.copy2(src_img, dst_img)
    shutil.copy2(src_lbl, dst_lbl)

    copy_records.append({
        "split": split_name,
        "canonical_class": row.canonical_class,
        "source_clean_image_path": str(src_img),
        "source_clean_label_path": str(src_lbl),
        "yolo_image_path": str(dst_img),
        "yolo_label_path": str(dst_lbl),
        "filename": src_img.name,
    })

yolo_copy_df = pd.DataFrame(copy_records)
save_df(yolo_copy_df, REPORT_ROOT / "17_yolo_dataset_manifest.csv")

dataset_yaml = {
    "path": str(YOLO_OUT),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": YOLO_ID_TO_CLASS
}

with open(YOLO_OUT / "dataset.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(dataset_yaml, f, sort_keys=False, allow_unicode=True)

print("=== YOLO SPLIT SUMMARY ===")
print("YOLO split method:", yolo_split_method)
display(yolo_copy_df.groupby(["split", "canonical_class"]).size().reset_index(name="num_images"))

print("\nSaved dataset.yaml to:", YOLO_OUT / "dataset.yaml")
with open(YOLO_OUT / "dataset.yaml", "r", encoding="utf-8") as f:
    print(f.read())

Copy YOLO split files:   0%|          | 0/1707 [00:00<?, ?it/s]

=== YOLO SPLIT SUMMARY ===
YOLO split method: stratified_train_test_split


,split,canonical_class,num_images
0,test,Anthracnose,54
1,test,BacterialSpot,69
2,test,Curl,54
3,test,RingSpot,80
4,train,Anthracnose,248
5,train,BacterialSpot,320
6,train,Curl,253
7,train,RingSpot,373
8,val,Anthracnose,53
9,val,BacterialSpot,69



Saved dataset.yaml to: /content/drive/MyDrive/Papaya_Leaf_Disease_Classification/prepared_data_v1/yolo_dataset/dataset.yaml
path: /content/drive/MyDrive/Papaya_Leaf_Disease_Classification/prepared_data_v1/yolo_dataset
train: images/train
val: images/val
test: images/test
names:
  0: Anthracnose
  1: BacterialSpot
  2: Curl
  3: RingSpot



In [11]:
# CELL 10: tạo CNN dataset từ crop bệnh + ảnh Healthy
# Mục đích:
# - Crop bbox bệnh từ cleaned images theo split YOLO để tránh leakage
# - Healthy chia train/val/test riêng
# - Xuất dataset classification cho CNN

if REBUILD_OUTPUT_DIRS:
    reset_dir(CNN_OUT)
else:
    ensure_dir(CNN_OUT)

for split_name in ["train", "val", "test"]:
    for cls_name in ALL_CLASSES:
        ensure_dir(CNN_OUT / split_name / cls_name)

cnn_records = []
cnn_issues = []

# ===== A. CROP disease images theo split YOLO =====
for row in tqdm(yolo_split_df.itertuples(index=False), total=len(yolo_split_df), desc="Create CNN disease crops"):
    cls_name = row.canonical_class
    split_name = row.split
    img_path = Path(row.clean_image_path)
    lbl_path = Path(row.clean_label_path)

    try:
        with Image.open(img_path) as img:
            img = img.convert("RGB")
            img_w, img_h = img.size
            label_rows = read_clean_label_rows(lbl_path)

            if len(label_rows) == 0:
                cnn_issues.append({
                    "issue_type": "empty_clean_label_when_cropping",
                    "canonical_class": cls_name,
                    "split": split_name,
                    "image_path": str(img_path),
                    "label_path": str(lbl_path),
                })
                continue

            crop_idx = 0
            for cls_id, x, y, w, h in label_rows:
                crop_idx += 1

                x1, y1, x2, y2 = yolo_to_pixel_box(
                    x, y, w, h,
                    img_w=img_w,
                    img_h=img_h,
                    expand_ratio=CNN_BBOX_EXPAND_RATIO
                )

                crop_w = x2 - x1
                crop_h = y2 - y1

                if crop_w < CNN_MIN_CROP_SIZE or crop_h < CNN_MIN_CROP_SIZE:
                    cnn_issues.append({
                        "issue_type": "crop_too_small",
                        "canonical_class": cls_name,
                        "split": split_name,
                        "image_path": str(img_path),
                        "label_path": str(lbl_path),
                        "detail": f"crop_size=({crop_w},{crop_h})"
                    })
                    continue

                crop_img = img.crop((x1, y1, x2, y2))
                crop_name = f"{img_path.stem}_crop{crop_idx:02d}{CNN_CROP_SAVE_EXT}"
                crop_path = CNN_OUT / split_name / cls_name / crop_name
                crop_img.save(crop_path, quality=JPEG_QUALITY)

                cnn_records.append({
                    "split": split_name,
                    "canonical_class": cls_name,
                    "sample_type": "disease_crop",
                    "source_clean_image_path": str(img_path),
                    "source_clean_label_path": str(lbl_path),
                    "cnn_sample_path": str(crop_path),
                    "crop_index": crop_idx,
                    "crop_width": crop_w,
                    "crop_height": crop_h,
                })

    except Exception as e:
        cnn_issues.append({
            "issue_type": "disease_crop_exception",
            "canonical_class": cls_name,
            "split": split_name,
            "image_path": str(img_path),
            "label_path": str(lbl_path),
            "detail": str(e)
        })

# ===== B. Healthy split =====
clean_healthy_df = clean_manifest_df[clean_manifest_df["canonical_class"] == "Healthy"].copy().reset_index(drop=True)

if len(clean_healthy_df) > 0:
    healthy_train_df, healthy_val_df, healthy_test_df = simple_random_split_single_class(
        clean_healthy_df,
        train_ratio=TRAIN_RATIO,
        val_ratio=VAL_RATIO,
        test_ratio=TEST_RATIO,
        seed=SEED
    )

    healthy_train_df["split"] = "train"
    healthy_val_df["split"]   = "val"
    healthy_test_df["split"]  = "test"

    healthy_split_df = pd.concat([healthy_train_df, healthy_val_df, healthy_test_df], ignore_index=True)

    for row in tqdm(healthy_split_df.itertuples(index=False), total=len(healthy_split_df), desc="Copy Healthy for CNN"):
        src_img = Path(row.clean_image_path)
        dst_img = CNN_OUT / row.split / "Healthy" / src_img.name
        shutil.copy2(src_img, dst_img)

        cnn_records.append({
            "split": row.split,
            "canonical_class": "Healthy",
            "sample_type": "healthy_image",
            "source_clean_image_path": str(src_img),
            "source_clean_label_path": None,
            "cnn_sample_path": str(dst_img),
            "crop_index": None,
            "crop_width": row.width,
            "crop_height": row.height,
        })

    save_df(healthy_split_df, REPORT_ROOT / "18_healthy_split_manifest.csv")
else:
    healthy_split_df = pd.DataFrame()
    print("Cảnh báo: không có ảnh Healthy sạch để tạo CNN healthy split.")

cnn_manifest_df = pd.DataFrame(cnn_records)
cnn_issues_df   = pd.DataFrame(cnn_issues)

save_df(cnn_manifest_df, REPORT_ROOT / "19_cnn_dataset_manifest.csv")
save_df(cnn_issues_df, REPORT_ROOT / "20_cnn_dataset_issues.csv")

print("=== CNN DATASET SUMMARY ===")
display(cnn_manifest_df.groupby(["split", "canonical_class"]).size().reset_index(name="num_samples"))

if len(cnn_issues_df) > 0:
    print("\nCNN issues (top 20):")
    display(cnn_issues_df.head(20))

Create CNN disease crops:   0%|          | 0/1707 [00:00<?, ?it/s]

Copy Healthy for CNN:   0%|          | 0/228 [00:00<?, ?it/s]

=== CNN DATASET SUMMARY ===


,split,canonical_class,num_samples
0,test,Anthracnose,1040
1,test,BacterialSpot,69
2,test,Curl,54
3,test,Healthy,34
4,test,RingSpot,639
5,train,Anthracnose,4416
6,train,BacterialSpot,320
7,train,Curl,254
8,train,Healthy,160
9,train,RingSpot,2412



CNN issues (top 20):


,issue_type,canonical_class,split,image_path,label_path,detail
0,crop_too_small,Anthracnose,train,/content/drive/MyDrive/Papaya_Leaf_Disease_Cla...,/content/drive/MyDrive/Papaya_Leaf_Disease_Cla...,"crop_size=(11,7)"
1,crop_too_small,RingSpot,train,/content/drive/MyDrive/Papaya_Leaf_Disease_Cla...,/content/drive/MyDrive/Papaya_Leaf_Disease_Cla...,"crop_size=(22,7)"
2,crop_too_small,Anthracnose,train,/content/drive/MyDrive/Papaya_Leaf_Disease_Cla...,/content/drive/MyDrive/Papaya_Leaf_Disease_Cla...,"crop_size=(7,8)"
3,crop_too_small,Anthracnose,train,/content/drive/MyDrive/Papaya_Leaf_Disease_Cla...,/content/drive/MyDrive/Papaya_Leaf_Disease_Cla...,"crop_size=(7,7)"
4,crop_too_small,Anthracnose,train,/content/drive/MyDrive/Papaya_Leaf_Disease_Cla...,/content/drive/MyDrive/Papaya_Leaf_Disease_Cla...,"crop_size=(7,8)"
5,crop_too_small,Anthracnose,train,/content/drive/MyDrive/Papaya_Leaf_Disease_Cla...,/content/drive/MyDrive/Papaya_Leaf_Disease_Cla...,"crop_size=(11,7)"
6,crop_too_small,Anthracnose,train,/content/drive/MyDrive/Papaya_Leaf_Disease_Cla...,/content/drive/MyDrive/Papaya_Leaf_Disease_Cla...,"crop_size=(7,8)"
7,crop_too_small,Anthracnose,train,/content/drive/MyDrive/Papaya_Leaf_Disease_Cla...,/content/drive/MyDrive/Papaya_Leaf_Disease_Cla...,"crop_size=(10,7)"
8,crop_too_small,Anthracnose,train,/content/drive/MyDrive/Papaya_Leaf_Disease_Cla...,/content/drive/MyDrive/Papaya_Leaf_Disease_Cla...,"crop_size=(6,6)"
9,crop_too_small,Anthracnose,train,/content/drive/MyDrive/Papaya_Leaf_Disease_Cla...,/content/drive/MyDrive/Papaya_Leaf_Disease_Cla...,"crop_size=(10,4)"


In [12]:
# CELL 11: báo cáo cuối notebook 1
# Mục đích:
# - Tổng hợp toàn bộ thống kê quan trọng
# - In ra màn hình và lưu JSON/CSV trên Drive

summary = {}

# 1) ảnh ban đầu từng lớp
initial_counts = (
    images_df.groupby("canonical_class")
    .size()
    .to_dict()
    if len(images_df) else {}
)
summary["initial_image_counts_by_class"] = initial_counts

# 2) số ảnh conflict giữa class
summary["cross_class_name_conflict_images"] = int(len(name_conflicts_df))
summary["cross_class_sha1_conflict_images"] = int(len(sha_conflicts_df))
summary["cross_class_phash_review_images"]  = int(len(phash_review_df))

# 3) số ảnh bị loại
summary["num_auto_excluded_source_images"] = int(len(auto_exclude_df))
summary["num_clean_skip_records"] = int(len(clean_skip_df))

# 4) số label lỗi
invalid_source_label_count = 0
if len(clean_skip_df) > 0:
    invalid_source_label_count = int((clean_skip_df["issue_type"] == "invalid_source_label").sum())
summary["num_invalid_source_labels"] = invalid_source_label_count

invalid_clean_label_count = 0
if len(clean_label_validation_df) > 0:
    invalid_clean_label_count = int((clean_label_validation_df["is_valid"] == False).sum())
summary["num_invalid_clean_labels"] = invalid_clean_label_count

# 5) số mẫu YOLO train/val/test
if len(yolo_copy_df) > 0:
    yolo_counts = (
        yolo_copy_df.groupby(["split", "canonical_class"])
        .size()
        .reset_index(name="count")
    )
    summary["yolo_counts_by_split_and_class"] = yolo_counts.to_dict(orient="records")
else:
    summary["yolo_counts_by_split_and_class"] = []

# 6) số mẫu CNN train/val/test
if len(cnn_manifest_df) > 0:
    cnn_counts = (
        cnn_manifest_df.groupby(["split", "canonical_class"])
        .size()
        .reset_index(name="count")
    )
    summary["cnn_counts_by_split_and_class"] = cnn_counts.to_dict(orient="records")
else:
    summary["cnn_counts_by_split_and_class"] = []

# 7) thông tin thêm
summary["yolo_split_method"] = yolo_split_method
summary["timestamp"] = time.strftime("%Y-%m-%d %H:%M:%S")
summary["project_root"] = str(PROJECT_ROOT)
summary["prep_root"] = str(PREP_ROOT)
summary["clean_root"] = str(CLEAN_ROOT)
summary["yolo_out"] = str(YOLO_OUT)
summary["cnn_out"] = str(CNN_OUT)
summary["report_root"] = str(REPORT_ROOT)

save_json(summary, REPORT_ROOT / "21_notebook1_summary.json")

print("========== NOTEBOOK 1 FINAL SUMMARY ==========")
print(json.dumps(summary, indent=2, ensure_ascii=False))

print("\n=== CLEAN COUNTS BY CLASS ===")
if len(clean_manifest_df) > 0:
    display(clean_manifest_df.groupby("canonical_class").size().reset_index(name="num_clean_samples"))

print("\n=== YOLO COUNTS BY SPLIT/CLASS ===")
if len(yolo_copy_df) > 0:
    display(yolo_copy_df.groupby(["split", "canonical_class"]).size().reset_index(name="num_images"))

print("\n=== CNN COUNTS BY SPLIT/CLASS ===")
if len(cnn_manifest_df) > 0:
    display(cnn_manifest_df.groupby(["split", "canonical_class"]).size().reset_index(name="num_samples"))

print("\nĐã lưu toàn bộ manifest / issue logs / summary vào:")
print(REPORT_ROOT)

========== NOTEBOOK 1 FINAL SUMMARY ==========
{
  "initial_image_counts_by_class": {
    "Anthracnose": 355,
    "BacterialSpot": 458,
    "Curl": 585,
    "Healthy": 228,
    "RingSpot": 533
  },
  "cross_class_name_conflict_images": 0,
  "cross_class_sha1_conflict_images": 448,
  "cross_class_phash_review_images": 448,
  "num_auto_excluded_source_images": 224,
  "num_clean_skip_records": 224,
  "num_invalid_source_labels": 0,
  "num_invalid_clean_labels": 158,
  "yolo_counts_by_split_and_class": [
    {
      "split": "test",
      "canonical_class": "Anthracnose",
      "count": 54
    },
    {
      "split": "test",
      "canonical_class": "BacterialSpot",
      "count": 69
    },
    {
      "split": "test",
      "canonical_class": "Curl",
      "count": 54
    },
    {
      "split": "test",
      "canonical_class": "RingSpot",
      "count": 80
    },
    {
      "split": "train",
      "canonical_class": "Anthracnose",
      "count": 248
    },
    {
      "split": "train",


,canonical_class,num_clean_samples
0,Anthracnose,355
1,BacterialSpot,458
2,Curl,361
3,Healthy,228
4,RingSpot,533



=== YOLO COUNTS BY SPLIT/CLASS ===


,split,canonical_class,num_images
0,test,Anthracnose,54
1,test,BacterialSpot,69
2,test,Curl,54
3,test,RingSpot,80
4,train,Anthracnose,248
5,train,BacterialSpot,320
6,train,Curl,253
7,train,RingSpot,373
8,val,Anthracnose,53
9,val,BacterialSpot,69



=== CNN COUNTS BY SPLIT/CLASS ===


,split,canonical_class,num_samples
0,test,Anthracnose,1040
1,test,BacterialSpot,69
2,test,Curl,54
3,test,Healthy,34
4,test,RingSpot,639
5,train,Anthracnose,4416
6,train,BacterialSpot,320
7,train,Curl,254
8,train,Healthy,160
9,train,RingSpot,2412



Đã lưu toàn bộ manifest / issue logs / summary vào:
/content/drive/MyDrive/Papaya_Leaf_Disease_Classification/prepared_data_v1/reports
